# NYT10m test: supplied-entity encoders and majority ensemble

This notebook uses the official manually annotated NYT10m test release. A bag
contains sentences for one **ordered head → tail entity pair**; the output is a set
of supported positive relation IDs, or an empty set (`NA`). Different sentences
may support different relations for the same pair.

The release contains 11,086 rows encoding 9,744 distinct sentence/mention units:
a sentence with several manual labels is repeated once per label. These rows are
merged by exact text, ordered IDs and supplied spans before selecting at most
20 distinct sentence/mention units per bag. Selection is evenly spaced in first
occurrence order and never uses labels or predictions.

The scoring reference is the union of manual positive labels **only for the
selected sentences**. No database facts or omitted-sentence labels are added.
The original file is unchanged. The verified release reconstructs 5,174 bags;
37 bags are shortened, leaving 9,001 selected sentences. In this release the
selected and full bag references happen to coincide (3,899 positive facts).
This coincidence is checked rather than assumed for other datasets.

`nyt10m_test_protocol.py` handles reconstruction, provenance and scoring.
The notebooks contain execution settings and stage calls. Test results do not
select thresholds, prompts, reasoning effort or ensemble rules.


By default, Run All displays the published, fingerprint-checked summaries without local datasets, model weights or paid calls. Enable local analysis or execution explicitly to verify or extend the retained local run.


## 1. Environment and execution flags

Default Run All displays published reports using an analysis environment, without local datasets or checkpoints. Set `ANALYZE_LOCAL_RESULTS=True` to verify and rescore retained local caches without new inference. Set `RUN_LOCAL_INFERENCE=True` to perform offline checks and inference for enabled members. Local stages require the project `.venv` kernel. Enable `PREFETCH_ASSETS` explicitly while connected if model assets are missing; downloads precede offline loading. Existing validation runs are never overwritten.


In [2]:
# Public report mode needs only the repository and its analysis dependencies.
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'source/nyt10m_test_protocol.py').is_file())
sys.path.insert(0, str(ROOT / 'source'))
from publication_utils import load_test_report
from nyt10m_test_protocol import scorer_self_check

RUN_LOCAL_INFERENCE = False
ANALYZE_LOCAL_RESULTS = False
RUN_BASE = True
RUN_LARGE = True
RUN_GLIDRE = True
PREFETCH_ASSETS = False
PREVENT_SLEEP = True
MEMBERS = {'base': RUN_BASE, 'large': RUN_LARGE, 'glidre': RUN_GLIDRE}
USE_LOCAL_RESULTS = RUN_LOCAL_INFERENCE or ANALYZE_LOCAL_RESULTS or PREFETCH_ASSETS
EXPERIMENT = None
if USE_LOCAL_RESULTS and Path(sys.prefix).resolve() != (ROOT / '.venv').resolve():
    raise RuntimeError('Select the project .venv kernel for local preparation or inference.')
print('Mode:', 'local experiment' if USE_LOCAL_RESULTS else 'published reports only')


Mode: published reports only


## 2. Published identity or local frozen-input preparation

Public mode verifies the compact report against the published frozen specification. With local analysis or execution enabled, preparation verifies the released test hash, exact mention spans, label inventory, manual reconstruction and historical adapter hashes. It persists separate inputs/references and the versioned specification. An optional asset stage identifies checkpoint directories and fetches missing enabled assets before offline execution. Changed settings stop cache reuse.


In [3]:
# Prepare frozen local evidence only when local analysis or execution is enabled.
if USE_LOCAL_RESULTS:
    from nyt10m_test_protocol import prepare_experiment
    from nyt10m_test_encoders import prefetch_assets, run_local, encoder_status, report_slm
    display(pd.DataFrame(prefetch_assets(PREFETCH_ASSETS)))
    EXPERIMENT = prepare_experiment()
    display(pd.DataFrame([EXPERIMENT['audit']]))
    print('Run directory:', EXPERIMENT['run'])
    print('Frozen cutoff settings:', EXPERIMENT['settings']['cutoffs'])
else:
    SLM_REPORT = load_test_report(ROOT, 'test_slm_summary.json')
    print('Published run fingerprint:', SLM_REPORT['fingerprint'])
    print('Local data preparation and downloads disabled.')


Published run fingerprint: b7ab3463602ffb361ff4ecdbc37ce947b0bca7b9cbcfa0221e7761ab3de3f6a3
Local data preparation and downloads disabled.


## 3. Constructed scorer checks

These checks use artificial labels and scores, including missed positives,
false positives, `NA`, invalid answers, duplicate labels, strict threshold
boundaries and two-of-three voting. No checkpoint or API call is involved.


In [4]:
# Verify deterministic scoring before any optional test inference.
print(scorer_self_check())


{'success': True, 'scope': 'synthetic scoring/threshold/voting cases; no model or API calls'}


## 4. Offline supplied-span preflight

Each enabled checkpoint loads in an isolated subprocess, one at a time. The
preflight verifies native relation-layer parity, supplied head/tail direction,
absence of entity recognition, every selected mention boundary and the longest
actual input. GLiDRE uses its pinned isolated helper runtime. Successful receipts
are bound to the shared experiment fingerprint and can be reused unchanged.
This stage performs model forward passes only when execution is enabled.


In [5]:
# Runtime model/input preflight occurs only with explicit inference enabled.
if RUN_LOCAL_INFERENCE:
    run_local(EXPERIMENT, MEMBERS, preflight_only=True,
              enabled=True, prevent_sleep=PREVENT_SLEEP)
else:
    print('Checkpoint loading and inference preflight disabled.')


Checkpoint loading and inference preflight disabled.


## 5. Sequential inference and resumable score caches

Base, large and GLiDRE run sequentially in separate CPU processes. The existing
adapters score the supplied ordered mentions without entity recognition. All 24
scores per selected sentence and bag-wise maxima are saved. The fixed strict
cutoffs are base > 0.5, large > 0.7 and GLiDRE > 0.7.

Complete caches are verified and skipped. Failed computation remains pending;
three consecutive failures or less than 2 GiB free disk space stop the worker.
Repeated execution resumes work. `caffeinate` prevents idle sleep when enabled
and available on macOS.


In [6]:
# Generate missing scores only when explicitly enabled; completed caches resume.
if RUN_LOCAL_INFERENCE:
    run_local(EXPERIMENT, MEMBERS, enabled=True, prevent_sleep=PREVENT_SLEEP)
else:
    print('Local inference disabled; published results are retained.')


Local inference disabled; published results are retained.


## 6. Completeness and the two primary local systems

Final scoring requires valid complete score caches for all three members.
The primary table contains **large @ 0.7** and **majority base @ 0.5 / large @ 0.7 /
GLiDRE @ 0.7**. Base/GLiDRE have no separate test leaderboard rows; their saved
outputs and validation evidence provide provenance. Each relation receives an
independent vote, and at least two members must accept it.

Precision, recall and micro-F1 use selected-sentence manual references. Ensemble
time includes all three member inferences, even though large's computation is
shared with the standalone comparison. Checkpoint loading and preflight are
separate from reported inference time. No test-based threshold sweep is run.


In [7]:
# Local verification requires complete caches; public mode reads compact summaries.
if USE_LOCAL_RESULTS:
    STATUS = encoder_status(EXPERIMENT)
    display(pd.DataFrame(STATUS))
    if all(row['valid'] == row['expected_bags'] for row in STATUS):
        SLM_REPORT = report_slm(EXPERIMENT)
    else:
        raise RuntimeError('The local test is incomplete; no final quality score is reported.')
else:
    SLM_REPORT = load_test_report(ROOT, 'test_slm_summary.json')
with pd.option_context('display.max_columns', None):
    display(pd.DataFrame(SLM_REPORT['rows']))
print('Compact report: results/test_slm_summary.json')


,FN,FP,TP,bags,invalid,invalid_rate,mean_wall_s,micro_f1,positive_predictions_on_reference_empty_bags,precision,recall,reference_empty_bags,successful_exact_match_bags,system,timing_scope,total_wall_s
0,1973,1543,1926,5174,0,0.0,0.378337,0.522801,526,0.555203,0.493973,1855,2410,large_0.7,sum of CPU bag attempts plus session label enc...,1957.517133
1,2366,688,1533,5174,0,0.0,0.735194,0.500980,308,0.690230,0.393178,1855,2588,majority_0.5_0.7_0.7,sum of CPU bag attempts plus session label enc...,3803.894352


Compact report: results/test_slm_summary.json


## 7. Shared comparison after the API notebook completes

This cell reads the completed SLM and LLM reports only. Both must match the same
input/reference fingerprint. CPU inference time and API request time include
different operations and are labelled separately. No inference occurs here.


In [8]:
# Display the three predeclared systems from the matching published comparison.
COMBINED_REPORT = load_test_report(ROOT, 'test_comparison_summary.json')
with pd.option_context('display.max_columns', None):
    display(pd.DataFrame(COMBINED_REPORT['rows']))
print(COMBINED_REPORT['timing_note'])


,FN,FP,TP,bags,invalid,invalid_rate,mean_wall_s,micro_f1,positive_predictions_on_reference_empty_bags,precision,recall,reference_empty_bags,successful_exact_match_bags,system,timing_scope,total_wall_s,token_charge_USD_estimate
0,1973,1543,1926,5174,0,0.000000,0.378337,0.522801,526,0.555203,0.493973,1855,2410,large_0.7,sum of CPU bag attempts plus session label enc...,1957.517133,NaN
1,2366,688,1533,5174,0,0.000000,0.735194,0.500980,308,0.690230,0.393178,1855,2588,majority_0.5_0.7_0.7,sum of CPU bag attempts plus session label enc...,3803.894352,NaN
2,1721,390,2178,5174,78,0.015075,4.256658,0.673574,270,0.848131,0.558605,1855,3317,deepseek_low,sum of API request attempts; includes service/...,22023.948629,2.409668


local CPU member sums and API network/service latency are distinct measurements


## Saved-result statistical analysis

[Test_results_analysis.ipynb](Test_results_analysis.ipynb) contains the completed paired bag-bootstrap F1 intervals, quality–latency and precision/recall figures, relation-level voting diagnostics and separate cost dimensions. It uses saved predictions only; default Run All does not load models or call an API.
